# NeutralMotion Schedule Tutorial

This notebook shows how to run `naive_dag` and `naive_n_dag` to produce schedule files from QASM circuits or a `step_order` text file. It covers installation, direct Python function calls, and CLI equivalents. 

## Installation

You can use the project from inside a local clone or install it directly from GitHub.

From inside the repository root:

```bash
pip install -e .
```

Directly from GitHub:

```bash
pip install git+https://github.com/Gerwinlab/NeutralMotion.git
```

If you have not installed the package yet but you are running this notebook from the repository root, the next cell adds `src/` to `sys.path` so the imports still work.

In [1]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from naive_dag.main import main as naive_dag_main
from naive_n_dag.main import load_config as load_naive_n_config
from naive_n_dag.main import main as naive_n_dag_main

In [2]:
qasm_dir = repo_root / "inputs" / "qasm_files"
output_dir = repo_root / "outputs" / "tutorial_runs"
output_dir.mkdir(parents=True, exist_ok=True)

example_qasm = qasm_dir / "qft_6.qasm"
example_qasm

PosixPath('/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft_6.qasm')

## Running `naive_dag`

`naive_dag.main(...)` expects:

- a config mapping
- a QASM filename or path
- an optional schedule output directory
- optional `quiet` and `seed` arguments

Any JSON config that uses `"fill_strategy": "random"` is also sufficient for `naive_dag`. The first example below reuses `inputs/algorithms/bb144_n_random.json`, and the second example shows how to define a small config inline.

In [3]:
naive_dag_config_path = repo_root / "inputs" / "algorithms" / "bb144_n_random.json"
naive_dag_config = json.loads(naive_dag_config_path.read_text())

naive_dag_exit_code = naive_dag_main(
    naive_dag_config,
    str(example_qasm),
    schedule_output_dir=output_dir,
    quiet=False,
    seed=0,
)

naive_dag_schedule = output_dir / "qft_6.schedule.txt"
print("exit_code:", naive_dag_exit_code)
print("schedule:", naive_dag_schedule)

naive Dag config loaded
dimensions=[17, 17] num_NA=288 qasm_dir=../../../qasm_outputs
qasm_file=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft_6.qasm
31726.676923076924 microsecond
schedule_file=/home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_6.schedule.txt
exit_code: 0
schedule: /home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_6.schedule.txt


### Inline `naive_dag` config example

If you want a minimal self-contained example, you can also define the config directly in Python.

In [4]:
naive_dag_inline_config = {
    "dimensions": [3, 3],
    "num_NA": 6,
    "qasm_base_dir": str(qasm_dir),
    "transfer_SLM_AOD": "60 microseconds",
    "max_acceleration": "2500 m/s^2",
    "max_velocity": "0.13 m/s",
    "rydberg_radius": "15 micrometers",
    "average_single_gate_time": "0.1 microseconds",
    "average_two_gate_time": "0.5 microseconds",
    "t_switch": "1.5 microseconds",
    "jerk": "constant",
}

naive_dag_inline_exit_code = naive_dag_main(
    naive_dag_inline_config,
    "qft_6.qasm",
    schedule_output_dir=output_dir,
    quiet=False,
    seed=1,
)

print("exit_code:", naive_dag_inline_exit_code)
print("schedule:", output_dir / "qft_6.schedule.txt")

naive Dag config loaded
dimensions=[3, 3] num_NA=6 qasm_dir=/home/gage/Quantum_Class/Phys765/inputs/qasm_files
qasm_file=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft_6.qasm
6480.215384615385 microsecond
schedule_file=/home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_6.schedule.txt
exit_code: 0
schedule: /home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_6.schedule.txt


In [5]:
if naive_dag_schedule.exists():
    print("\n".join(naive_dag_schedule.read_text().splitlines()[:20]))
else:
    print(f"Schedule file not found yet: {naive_dag_schedule}")

qasm_file: qft_6.qasm
final_time: 6480.215384615385 microsecond
lattice_spacing (rydberg_radius): 14.999999999999998 micrometer
random number seed: 1
T=0
initialize q[0] -> (0,4); q[1] -> (0,2); q[2] -> (2,4); q[3] -> (0,0); q[4] -> (2,0); q[5] -> (2,2);
T=1
h q[5];
T=2
load q[5] -> (2,2) : (2,1)
T=3
cp(pi/2) q[5],q[4];
T=4
h q[4];
T=5
move q[5] (2,1) : (0,1)
T=6
cp(pi/4) q[5],q[3];
T=7
move q[5] (0,1) : (1,1)


## Running `naive_n_dag`

`naive_n_dag` already has example configs in `inputs/algorithms/`. This version supports extra options such as:

- `output_name` to control the filename
- `log=True` to write a FastSA CSV log when `fill_strategy` is `fastsa`
- `quiet=True` to suppress status printing
- `seed` to make random placement reproducible when the fill strategy uses randomness
- `step_order` in config to load a `.txt` timestep file (when present, the QASM argument is ignored)


In [6]:
naive_n_config_path = repo_root / "inputs" / "algorithms" / "qft_06_n_fastsa.json"
naive_n_config = load_naive_n_config(naive_n_config_path)

naive_n_exit_code = naive_n_dag_main(
    naive_n_config,
    "qft_6.qasm",
    schedule_output_dir=output_dir,
    config_path=naive_n_config_path,
    quiet=False,
    seed=0,
    output_name="qft_06_n_fastsa_tutorial",
    log=True,
)

naive_n_schedule = output_dir / "qft_06_n_fastsa_tutorial.schedule.txt"
naive_n_log = output_dir / "qft_06_n_fastsa_tutorial.fastsa_log.csv"
print("exit_code:", naive_n_exit_code)
print("schedule:", naive_n_schedule)
print("fastsa_log:", naive_n_log)

naive_n_dag config loaded
dimensions=[3, 3] num_NA=6 qasm_dir=/home/gage/Quantum_Class/Phys765/inputs/qasm_files
qasm_file=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft_6.qasm
max_dimension=[3, 3]
fill_strategy=fastsa
schedule_output_dir=/home/gage/Quantum_Class/Phys765/outputs/tutorial_runs
two_qubit_layers=9
scheduled_timesteps=60
estimated_time=11388.061538461538 microsecond
schedule_file=/home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_06_n_fastsa_tutorial.schedule.txt
fastsa_log_file=/home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_06_n_fastsa_tutorial.fastsa_log.csv
exit_code: 0
schedule: /home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_06_n_fastsa_tutorial.schedule.txt
fastsa_log: /home/gage/Quantum_Class/Phys765/outputs/tutorial_runs/qft_06_n_fastsa_tutorial.fastsa_log.csv


In [7]:
for path in [naive_n_schedule, naive_n_log]:
    print(f"{path.name}: {'present' if path.exists() else 'missing'}")

qft_06_n_fastsa_tutorial.schedule.txt: present
qft_06_n_fastsa_tutorial.fastsa_log.csv: present


## Using `step_order` Text Input

You can provide a text-based two-qubit gate order with the config key `step_order` (resolved relative to the JSON config file).

Behavior when `step_order` is provided:

- the QASM argument is ignored (warning emitted)
- only two-qubit gates are loaded from the text file (warning emitted)
- `single_layers` is returned as an array of empty layers sized to the two-qubit DAG depth (`layers + 1`)

Expected per-timestep format:

```text
<T>, <gate_index>: <q0> <q1>
... (one or more gate lines for the same T)
T = <T>, cx = <count_of_gate_lines>
```

* this only supports cx gates. The primary focus of this feature is for testing layer creation where each time step becomes a layer.


In [10]:
step_order_config_path = repo_root / "inputs" / "algorithms" / "qft_n63_init_order.json"
step_order_config = load_naive_n_config(step_order_config_path)

step_order_value = step_order_config["step_order"]
step_order_path = Path(step_order_value)
if not step_order_path.is_absolute():
    step_order_path = (step_order_config_path.parent / step_order_path).resolve()

print("config:", step_order_config_path)
print("step_order:", step_order_path)

with step_order_path.open("r", encoding="utf-8") as f:
    for _ in range(12):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())


config: /home/gage/Quantum_Class/Phys765/inputs/algorithms/qft_n63_init_order.json
step_order: /home/gage/Quantum_Class/Phys765/inputs/qasm_files/init_order.txt
0, 0: 0 1
T = 0, cx = 1
1, 1: 0 1
T = 1, cx = 1
2, 2: 0 2
T = 2, cx = 1
3, 3: 0 2
T = 3, cx = 1
4, 4: 1 2
4, 6: 0 3
T = 4, cx = 2
5, 5: 1 2


Example command with this mode:

```bash
PYTHONPATH=src python3 -m naive_n_dag inputs/algorithms/qft_n63_init_order.json inputs/qasm_files/qft_n63_transpiled.qasm outputs --log
```

The QASM argument is still required by the CLI signature, but it is ignored when `step_order` is present in config.


## CLI equivalents

If you prefer the command line, the same runs look like this.

```bash
python -m naive_dag inputs/algorithms/your_config.json qft_6.qasm outputs/tutorial_runs --seed 0
```

```bash
python -m naive_n_dag inputs/algorithms/qft_06_n_fastsa.json qft_6.qasm outputs/tutorial_runs --seed 0 --output-name qft_06_n_fastsa_tutorial --log
```

```bash
PYTHONPATH=src python3 -m naive_n_dag inputs/algorithms/qft_n63_init_order.json inputs/qasm_files/qft_n63_transpiled.qasm outputs --log
```

Useful flags:

- `--quiet` suppresses non-error output
- `--dump-config` prints the resolved config and exits
- `--seed` sets the placement seed
- `--output-name` is available in `naive_n_dag`
- `--log` is available in `naive_n_dag` when using FastSA

- when `step_order` is present in config, the CLI qasm argument is ignored


# Schedule Output Example

Below is a short example of a generated schedule file. This is only a small excerpt, not the full `schedule.txt`.

```text
qasm_file: qft_6.qasm
final_time: 6770.984615384615 microsecond
lattice_spacing (rydberg_radius): 14.999999999999998 micrometer
random number seed: 0

T=0
initialize q[0] -> (2,4); q[1] -> (2,0); q[2] -> (0,0); q[3] -> (0,4); q[4] -> (2,2); q[5] -> (0,2);
T=1
h q[5];
T=2
load q[5] -> (2,0) : (2,1)
T=3
cp(pi/2) q[5],q[4];
T=4
move q[5] (2,1) : (2,3)
T=5
unload q[5] -> (2,3) : (2,2)
```

## What the Header Means

- `qasm_file` is the input source recorded in the schedule header. With `step_order`, this will be the `.txt` step-order filename.
- `final_time` is the total estimated execution time of the full schedule.
- `lattice_spacing (rydberg_radius)` is the physical spacing used by the movement model.
- `random number seed` is the seed used for randomized placement when a random fill strategy is selected.

## What the Actions Mean

- `initialize` gives the fixed home position of each atom in the lattice. This is where atoms start, and it is also the position they return to after transport.
- `load` means an atom is transferred from the Spatial Light Modulator (SLM) into the 2D Acousto-Optic Deflector (AOD) transport system.
- The `load` time includes the SLM-to-AOD transfer time plus the motion required to enter the transport “highway” under the scheduler’s upper-bound timing model.
- `unload` is the reverse operation: the atom leaves the AOD transport path and returns to its fixed lattice site.
- `move q[5] (2,1) : (0,1)` means atom `q[5]` moves from lattice coordinate `(2,1)` to `(0,1)`.
- The user can infer the geometric distance from those coordinates, but the scheduler computes time internally from its predefined movement profile together with the configured `lattice_spacing`.
- Gate lines such as `h q[5];` and `cp(pi/2) q[5],q[4];` are emitted in an OpenQASM 3.0-style textual form.

## Gate Support

Currently, this scheduler supports only 1-qubit and 2-qubit gates.
